In [2]:
# basic idea is to split one document (for example) into larger chunks and then split these chunks into smaller chunks. You then embed these smaller chunks
# into a vector store. So you get accurate retrieval, plus enough context. 

# for tmo bonds use case:
# 1. larger chunks of meaningful components like topics, smaller chunks of meaningful components
# 2. larger chunks of fixed length, smaller chunks of fixed length
# 3. larger chunks of meaningful components, smaller chunks of fixed length

In [73]:
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.storage import InMemoryStore
from langchain.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
llm = ChatOpenAI(model="gpt-3.5-turbo-1106", temperature=0)
from langchain.retrievers import ParentDocumentRetriever
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
# This is the overall chain where we run these two chains in sequence.
from langchain.chains import SimpleSequentialChain, TransformChain
from langchain.chains import StuffDocumentsChain


In [74]:
# note: use openai == 0.28.1

In [75]:
import os
import openai

In [76]:
llm.predict("hello")

'Hello! How can I assist you today?'

In [6]:
loaders = [
    TextLoader("./paul_graham_essay.txt")
    #TextLoader("../../state_of_the_union.txt"),
]
docs = []
for l in loaders:
    docs.extend(l.load())

In [7]:
# This text splitter is used to create the parent documents
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)
# This text splitter is used to create the child documents
# It should create documents smaller than the parent
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="split_parents", embedding_function=OpenAIEmbeddings()
)
# The storage layer for the parent documents
store = InMemoryStore()

In [66]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [9]:
retriever.add_documents(docs)

In [199]:
# This is an LLMChain to transform query
transform_template = """Task: Extract the key words from a given human question or task, ignoring any additional stylistic or contextual instructions.

Example 1:
Input: "describe ABC as if you were a pirate"
Extracted subject: "ABC"

Example 2:
Input: "explain XYZ as if I were a 5-year-old"
Extracted subject: "XYZ"

Example 3:
Input: "How to do DFG, give me a brief summary"
Extracted subject: "DFG"

Given text: {text}
Extracted subject:

Given text: {text} 
Extracted subject:"""
synopsis_prompt_template = PromptTemplate(
    input_variables=["text"], template=transform_template
)
transform_chain = LLMChain(
    llm=llm, prompt=synopsis_prompt_template, output_key="query"
)

In [195]:
prompt_template = """Strictly use only the provided context below to answer the question that follows. 
If the context does not contain the necessary information, respond with 
"Sorry, I don't have enough information." 

Context:
{context}

Question:
{question}

Helpful Answer:"""
qn_prompt = PromptTemplate(
    template=prompt_template, input_variables=["context", "question"]
)

qn_chain = LLMChain(llm=llm, prompt=qn_prompt)


In [196]:
document_prompt = PromptTemplate(
    input_variables=["page_content"], template="Context:\n{page_content}"
)

qa_chain = StuffDocumentsChain(
    llm_chain=qn_chain,
    document_variable_name="context",
    document_prompt=document_prompt,
)

In [215]:
query = "explain paul's university journey as if you were a pirate"

In [216]:
transform_chain(query)['query']

'"paul\'s university journey"'

In [217]:
qa_chain({'input_documents':retriever.get_relevant_documents(transform_chain(query)['query']), 'question': query})

{'input_documents': [Document(page_content='I liked painting still lives because I was curious about what I was seeing. In everyday life, we aren\'t consciously aware of much we\'re seeing. Most visual perception is handled by low-level processes that merely tell your brain "that\'s a water droplet" without telling you details like where the lightest and darkest points are, or "that\'s a bush" without telling you the shape and position of every leaf. This is a feature of brains, not a bug. In everyday life it would be distracting to notice every leaf on every bush. But when you have to paint something, you have to look more closely, and when you do there\'s a lot to see. You can still be noticing new things after days of trying to paint something people usually take for granted, just as you can after days of trying to write an essay about something people usually take for granted.\n\nThis is not the only way to paint. I\'m not 100% sure it\'s even a good way to paint. But it seemed a g